In [ ]:
import pandas as pd
import numpy as np
import joblib

# 1. Load Model, Scaler, dan Dataset
best_model = joblib.load('../models/best_rf_model.pkl')
scaler = joblib.load('../models/scaler.pkl') # Memanggil kembali scaler dari proses training

df_train_clean = pd.read_csv('../data/processed/volve_train_clean.csv', sep=';', decimal=',')
df_impute_clean = pd.read_csv('../data/processed/volve_impute_clean.csv', sep=';', decimal=',')

feature_cols = [
    'BORE_OIL_VOL', 'BORE_GAS_VOL', 'BORE_WAT_VOL', 
    'AVG_DOWNHOLE_PRESSURE', 'AVG_WELLHEAD_PRESSURE', 'AVG_TEMPERATURE'
]

# === JURUS SAPU JAGAT ANTI-NaN ===
# Menambal sensor yang mungkin bolong di dataset impute
df_impute_clean[feature_cols] = df_impute_clean[feature_cols].bfill().ffill()

# 2. Eksekusi Normalisasi dan Imputasi
X_missing = df_impute_clean[feature_cols]
X_missing_scaled = scaler.transform(X_missing) # Normalisasi data bolong

# Suruh model menebak nilai choke
predicted_choke = best_model.predict(X_missing_scaled)

# Mengisi kolom target yang kosong dan dibulatkan 2 desimal agar rapi
df_impute_clean['AVG_CHOKE_SIZE_P'] = np.round(predicted_choke, 2)

# 3. Penggabungan Kembali Dataset (Rekonstruksi)
df_final = pd.concat([df_train_clean, df_impute_clean], ignore_index=True)

# Di Notebook 03 (Langkah 3):
if 'DATEPRD' in df_final.columns:
    df_final['DATEPRD'] = pd.to_datetime(df_final['DATEPRD'], format='%Y-%m-%d %H.%M.%S')
    df_final = df_final.sort_values(by=['NPD_WELL_BORE_CODE', 'DATEPRD']).reset_index(drop=True)

# 4. Ekspor Hasil Akhir
export_path = '../data/processed/Volve_Dataset_Akhir_Sempurna.csv'
df_final.to_csv(export_path, index=False)

print(f"✅ Proses imputasi berhasil!")
print(f"Total {len(df_impute_clean)} baris data kosong telah direstorasi oleh AI.")
print(f"Dataset utuh berhasil diekspor ke: '{export_path}'")